In [1]:
import os
import pandas as pd
from fastai.tabular.all import *
import ipywidgets as widgets
from IPython.display import display

# 1. ФИКСИРАНЕ НА ПЪТЯ
# Търсене директно на файла в цялата входна директория на Kaggle
file_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'vehicles.csv':
            file_path = os.path.join(root, file)
            break

if file_path:
    print(f"✅ Файлът е намерен: {file_path}")
else:
    # Ако файлът не е намерен, търсене на най-вероятния път ръчно
    file_path = '/kaggle/input/craigslist-carstrucks-data/vehicles.csv'
    print(f"⚠️ Файлът не е намерен автоматично, опит с: {file_path}")

# 2. ЗАРЕЖДАНЕ И ПОЧИСТВАНЕ (т. 1.1 Python [cite: 3, 5])
cols = ['price', 'year', 'manufacturer', 'condition', 'fuel', 'odometer', 'transmission']
try:
    df = pd.read_csv(file_path, usecols=cols)
    # Премахване на екстремни стойности и празни редове
    df = df.dropna()
    df = df[(df.price > 1000) & (df.price < 100000)].reset_index(drop=True)
    # Използване на по-малка извадка за по-бърза работа в Jupyter (т. 1.2 )
    df = df.sample(n=20000, random_state=42) 
    print(f"📊 Заредени данни: {len(df)} реда.")
except Exception as e:
    print(f"❌ Грешка при четене на файла: {e}")

# 3. ПОДГОТОВКА НА МОДЕЛА (т. 1.4 FastAI [cite: 22, 24])
cat_names = ['manufacturer', 'condition', 'fuel', 'transmission']
cont_names = ['year', 'odometer']
procs = [Categorify, FillMissing, Normalize]

# Създаване на DataLoaders (т. 1.8 PyTorch основа )
dls = TabularDataLoaders.from_df(df, y_names="price", 
                                 cat_names=cat_names, 
                                 cont_names=cont_names, 
                                 procs=procs,
                                 bs=64)

# Обучение на модела
learn = tabular_learner(dls, metrics=mae)
print("🚀 Започва обучение на невронната мрежа...")
learn.fit_one_cycle(3)

# 4. ИНТЕРАКТИВНИ ОПЦИИ ЗА ПОТРЕБИТЕЛЯ
print("\n--- 🏎️ Изберете характеристики ---")

brand_w = widgets.Dropdown(options=sorted(df['manufacturer'].unique()), description='Марка:')
cond_w = widgets.Dropdown(options=sorted(df['condition'].unique()), description='Състояние:')
fuel_w = widgets.Dropdown(options=sorted(df['fuel'].unique()), description='Гориво:')
year_w = widgets.IntSlider(min=1990, max=2024, value=2015, description='Година:')
odo_w = widgets.IntText(value=80000, description='Пробег (мили):')
btn = widgets.Button(description='ПРЕДСКАЖИ ЦЕНА', button_style='success')
out = widgets.Output()

def on_click(b):
    with out:
        out.clear_output()
        # Създаване на тестов ред с данните от потребителя
        t_df = pd.DataFrame([{
            'manufacturer': brand_w.value,
            'condition': cond_w.value,
            'fuel': fuel_w.value,
            'transmission': 'automatic', # по подразбиране
            'year': year_w.value,
            'odometer': odo_w.value
        }])
        
        # --- 1. По-строго филтриране на данните ---
df = df[(df.price > 1500) & (df.price < 80000)].copy() # Премахване на "фалшивите" евтини коли
df['price'] = np.log1p(df['price']) # Логаритмично трансформираме цената

# --- 2. Обновени DataLoaders ---
dls = TabularDataLoaders.from_df(df, y_names="price", 
                                 cat_names=['manufacturer', 'condition', 'fuel'], 
                                 cont_names=['year', 'odometer'], 
                                 procs=[Categorify, FillMissing, Normalize],
                                 y_block=RegressionBlock()) # Указваме, че е регресия

# --- 3. Обучение с по-ниска скорост (Learning Rate) ---
learn = tabular_learner(dls, layers=[200,100], metrics=rmse)
learn.fit_one_cycle(5, 1e-2) # 5 епохи за по-добра прецизност [cite: 28]

# --- 4. Функция за предсказване ---
def on_click(b):
    with out:
        out.clear_output()
        t_df = pd.DataFrame([{
            'manufacturer': brand_w.value,
            'condition': cond_w.value,
            'fuel': fuel_w.value,
            'year': year_w.value,
            'odometer': odo_w.value
        }])
        
        dl = learn.dls.test_dl(t_df)
        preds, _ = learn.get_preds(dl=dl)
        
        final_res = np.expm1(preds[0].item())
        print(f"💰 Прогнозна цена: ${final_res:,.2f}")

btn.on_click(on_click)
display(brand_w, cond_w, fuel_w, year_w, odo_w, btn, out)

✅ Файлът е намерен: /kaggle/input/datasets/austinreese/craigslist-carstrucks-data/vehicles.csv
📊 Заредени данни: 20000 реда.
🚀 Започва обучение на невронната мрежа...


epoch,train_loss,valid_loss,mae,time
0,507669440.000000,513245664.000000,18609.367188,00:02
1,504603328.000000,512099808.000000,18597.509766,00:02
2,513934464.000000,511990688.000000,18595.330078,00:02



--- 🏎️ Изберете характеристики ---


epoch,train_loss,valid_loss,_rmse,time
0,4.983547,0.388352,0.623179,00:02
1,0.315620,0.239524,0.489412,00:02
2,0.264983,0.222880,0.472102,00:02
3,0.200161,0.224294,0.473597,00:02
4,0.183083,0.205929,0.453794,00:02


Dropdown(description='Марка:', options=('acura', 'alfa-romeo', 'aston-martin', 'audi', 'bmw', 'buick', 'cadill…

...

Dropdown(description='Гориво:', options=('diesel', 'electric', 'gas', 'hybrid', 'other'), value='diesel')

IntSlider(value=2015, description='Година:', max=2024, min=1990)

IntText(value=80000, description='Пробег (мили):')

Button(button_style='success', description='ПРЕДСКАЖИ ЦЕНА', style=ButtonStyle())

Output()

🏎️ Проект: Предсказване на цени на автомобили с Изкуствен Интелект Дисциплина: Изкуствен интелект в софтуерното инженерство 
Технологичен стек: Python, FastAI, PyTorch, Jupyter

    1. Въведение и подготовка на средата:
Целта на тази самостоятелна работа е разработването на модел за машинно обучение, който предсказва цената на употребявани автомобили на базата на реални данни от Craigslist.
        Среда за разработка: Използва се Jupyter Notebook в платформата Kaggle.
        Хардуерно ускорение: Използва се виртуална машина с Dedicated GPU, за да се оптимизира процесът на обучение на невронната мрежа.
        Библиотеки: Основната реализация стъпва на FastAI (построена върху PyTorch).

    2. Обработка на данните (Data Engineering)За качественото обучение на модела са приложени следните стъпки (Crash Course насоки):
        Почистване: Премахване на нереалистични цени (под $1,500 и над $80,000) за изчистване на "шума" в данните.
        Трансформация: Приложено е логаритмично мащабиране на целевата променлива (price), което е стандартна практика в регресионните модели за постигане на по-висока точност.
        Автоматизирани процеси (FastAI Procs):
            Categorify: Превръща текстовите категории в числови стойности.
            FillMissing: Автоматично справяне с липсващи данни.
            Normalize: Нормализиране на числовите променливи (година и пробег) за по-добър градиентен десен.

    3. Изграждане и обучение на модела
    Съгласно препоръките от FastAI Part 1, моделът е изграден чрез tabular_learner:
        Архитектура: Невронна мрежа с многослоен перцептрон, оптимизирана за таблични данни.
        Метод на обучение: Използван е fit_one_cycle, който динамично променя скоростта на учене (learning rate) за по-бърза и стабилна конвергенция.
        Метрика: Използва се MAE (Mean Absolute Error) за оценка на средното отклонение на предсказаната цена от реалната.

    4. Потребителски интерфейс и Инференция
    За да отговори на нуждите на софтуерното инженерство, проектът включва интерактивен модул:
        Използвани са: Jupyter Widgets за избор на марка, състояние, гориво и година.
        Потребителят получава мигновена прогноза чрез предварително обучената невронна мрежа.

    5. Заключение: Проектът демонстрира практическите приложения на Deep Learning библиотеките в реални софтуерни задачи.
        Използваните методи от FastAI и PyTorch позволяват бързо прототипиране и постигане на висока точност (State-of-the-art),дори при работа с големи и сложни масиви от данни.